# M5 Agentic AI - Customer Service Agent

## 1. Introduction
**Planning with code execution** means letting the LLM write code that *is* the plan. Compared to plain-text or JSON plans, this is more expressive and flexible: the code documents the steps **and** executes them.

We simulate a sunglasses store with an inventory and a set of transactions (sales, returns, balance updates). The LLM writes Python (TinyDB) code to query or update the records.

### 1.1 Lab Overview
- Create simple inventory and transaction datasets.
- Build a schema block describing the data.
- Prompt the LLM to write a plan as Python code (with comments explaining each step).
- Execute the code in a sandbox to obtain the answer.

### 1.2 Learning Outcomes
- Explain why letting the model write code (instead of JSON or plain-text plans) enables richer, more flexible planning.
- Prompt an LLM to produce Python code with step-by-step comments that both documents and executes the plan.
- Run the generated code in a controlled namespace and interpret the results.

> **Before you start:** copy `.env.example` (in the `CodeAsActionDemo` folder) to `.env` and add your `OPENAI_API_KEY`.

## 2. Setup

In [ ]:
# ==== Imports ====
from __future__ import annotations
import json
from dotenv import load_dotenv
from openai import OpenAI
import re, io, sys, traceback, json
from typing import Any, Dict, Optional
from tinydb import Query, where

# Utility modules
import utils      # helper functions for prompting/printing
import inv_utils  # functions for inventory, transactions, schema building, and TinyDB seeding

load_dotenv()
client = OpenAI()

`inv_utils` provides:
- `create_inventory()` – builds the sunglasses inventory.
- `create_transactions()` – builds the initial transaction log (opening balance).
- `seed_db()` – loads both into a JSON-backed TinyDB store (`store_db.json`).
- `build_schema_block()` – generates the schema description used in the prompt.
- `get_current_balance()` and `next_transaction_id()` – helpers the generated code uses for consistent updates.

### 2.1 Create Example Tables
[TinyDB](https://tinydb.readthedocs.io) is a lightweight document database in pure Python: data is stored as JSON, no server needed.

In [ ]:
db, inventory_tbl, transactions_tbl = inv_utils.seed_db()

Inspect the records in each table:

In [ ]:
utils.print_html(json.dumps(inventory_tbl.all(), indent=2), title="Inventory Table")
utils.print_html(json.dumps(transactions_tbl.all(), indent=2), title="Transactions Table")

**Inventory Table (`inventory_tbl`)**
- `item_id` (string): unique product identifier (e.g., SG001)
- `name` (string): style of sunglasses (e.g., Aviator, Classic)
- `description` (string): text description of the product
- `quantity_in_stock` (int): current stock available
- `price` (float): price in USD

**Transactions Table (`transactions_tbl`)**
- `transaction_id` (string): unique identifier (e.g., TXN001)
- `customer_name` (string): name of the customer, or OPENING_BALANCE for the initial entry
- `transaction_summary` (string): short description of the transaction
- `transaction_amount` (float): amount of money for this transaction
- `balance_after_transaction` (float): running balance after applying the transaction
- `timestamp` (string): ISO-8601 date/time

## Planning with Code Execution

### 2.1. The plan
The prompt instructs the model to **plan by writing code**: it explains each step in comments, then carries it out. It also makes the model decide whether the request is read-only or a state change, and enforces safe execution (no I/O, no network, TinyDB `Query` only, consistent mutations).

In [ ]:
PROMPT = """You are a senior data assistant. PLAN BY WRITING PYTHON CODE USING TINYDB.

Database Schema & Samples (read-only):
{schema_block}

Execution Environment (already imported/provided):
- Variables: db, inventory_tbl, transactions_tbl  # TinyDB Table objects
- Helpers: get_current_balance(tbl) -> float, next_transaction_id(tbl, prefix="TXN") -> str
- Natural language: user_request: str  # the original user message

PLANNING RULES (critical):
- Derive ALL filters/parameters from user_request (shape/keywords, price ranges "under/over/between", stock mentions,
  quantities, buy/return intent). Do NOT hard-code values.
- Build TinyDB queries dynamically with Query(). If a constraint isn't in user_request, don't apply it.
- Be conservative: if intent is ambiguous, do read-only (DRY RUN).

TRANSACTION POLICY (hard):
- Do NOT create aggregated multi-item transactions.
- If the request contains multiple items, create a separate transaction row PER ITEM.
- For each item:
  - compute its own line total (unit_price * qty),
  - insert ONE transaction with that amount,
  - update balance sequentially (balance += line_total),
  - update the item’s stock.
- If any requested item lacks sufficient stock, do NOT mutate anything; reply with STATUS="insufficient_stock".

HUMAN RESPONSE REQUIREMENT (hard):
- You MUST set a variable named `answer_text` (type str) with a short, customer-friendly sentence (1–2 lines).
- This sentence is the only user-facing message. No dataframes/JSON, no boilerplate disclaimers.
- If nothing matches, politely say so and offer a nearby alternative (closest style/price) or a next step.

ACTION POLICY:
- If the request clearly asks to change state (buy/purchase/return/restock/adjust):
    ACTION="mutate"; SHOULD_MUTATE=True; perform the change and write a matching transaction row.
  Otherwise:
    ACTION="read"; SHOULD_MUTATE=False; simulate and explain briefly as a dry run (in logs only).

FAILURE & EDGE-CASE HANDLING (must implement):
- Do not capture outer variables in Query.test. Pass them as explicit args.
- Always set a short `answer_text`. Also set a string `STATUS` to one of:
  "success", "no_match", "insufficient_stock", "invalid_request", "unsupported_intent".
- no_match: No items satisfy the filters → suggest the closest in style/price, or invite a different range.
- insufficient_stock: Item found but stock < requested qty → state available qty and offer the max you can fulfill.
- invalid_request: Unable to parse essential info (e.g., quantity for a purchase/return) → ask for the missing piece succinctly.
- unsupported_intent: The action is outside the store’s capabilities → provide the nearest supported alternative.
- In all cases, keep the tone helpful and concise (1–2 sentences). Put technical details (e.g., ACTION/DRY RUN) only in stdout logs.

OUTPUT CONTRACT:
- Return ONLY executable Python between these tags (no extra text):
  <execute_python>
  # your python
  </execute_python>

CODE CHECKLIST (follow in code):
1) Parse intent & constraints from user_request (regex ok).
2) Build TinyDB condition incrementally; query inventory_tbl.
3) If mutate: validate stock, update inventory, insert a transaction (new id, amount, balance, timestamp).
4) ALWAYS set:
   - `answer_text` (human sentence, required),
   - `STATUS` (see list above).
   Also print a brief log to stdout, e.g., "LOG: ACTION=read DRY_RUN=True STATUS=no_match".
5) Optional: set `answer_rows` or `answer_json` if useful, but `answer_text` is mandatory.

TONE EXAMPLES (for `answer_text`):
- success: "Yes, we have our Classic sunglasses, a round frame, for $60."
- no_match: "We don’t have round frames under $100 in stock right now, but our Moon round frame is available at $120."
- insufficient_stock: "We only have 1 pair of Classic left; I can reserve that for you."
- invalid_request: "I can help with that—how many pairs would you like to purchase?"
- unsupported_intent: "We can’t refurbish frames, but I can suggest similar new models."

Constraints:
- Use TinyDB Query for filtering. Standard library imports only if needed.
- Keep code clear and commented with numbered steps.

User request:
{question}
"""

### 2.2 From Prompt to Code (Planning in Code)
`generate_llm_code` builds a **live schema** from both tables, formats the prompt with the user's question, and asks the model for a plan-with-code response (an `<execute_python>` block). Nothing is executed here.

In [ ]:
def generate_llm_code(
    prompt: str,
    *,
    inventory_tbl,
    transactions_tbl,
    model: str = "gpt-4.1-mini",
    temperature: float = 0.2,
) -> str:
    """
    Ask the LLM to produce a plan-with-code response.
    Returns the FULL assistant content (including surrounding text and tags).
    The actual code extraction happens later in execute_generated_code.
    """
    schema_block = inv_utils.build_schema_block(inventory_tbl, transactions_tbl)
    prompt = PROMPT.format(schema_block=schema_block, question=prompt)

    resp = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {
                "role": "system",
                "content": "You write safe, well-commented TinyDB code to handle data questions and updates."
            },
            {"role": "user", "content": prompt},
        ],
    )
    content = resp.choices[0].message.content or ""

    return content

### 2.3 Try a Sample Prompt (Planning-in-Code)
Andrew's prompt from the lecture: **"Do you have any round sunglasses in stock that are under $100?"**

First, inspect the inventory manually for round frames (word-only match, so *wraparound* doesn't count).

In [ ]:
Item = Query()                    # Create a Query object to reference fields (e.g., Item.name, Item.description)

# Search the inventory table for documents where either the description OR the name
# contains the word "round" (case-insensitive). The check is done inline:
# - (v or "") ensures we handle None by converting it to an empty string
# - .lower() normalizes case
# - " round " enforces a crude word boundary (won't match "wraparound")
round_sunglasses = inventory_tbl.search(
    (Item.description.test(lambda v: " round " in ((v or "").lower()))) |
    (Item.name.test(        lambda v: " round " in ((v or "").lower())))
)

# Render the results as formatted JSON in the notebook UI
utils.print_html(json.dumps(round_sunglasses, indent=2), title="Inventory Status: Round Sunglasses")

There are two round styles in stock, but only one is under $100 — the **Classic** (SG005, $60). The Moon ($120) is round but too expensive.

Now let the model generate a plan in code (no execution yet):

In [ ]:
# Andrew's prompt from the lecture
prompt_round = "Do you have any round sunglasses in stock that are under $100?"

# Generate the plan-as-code (FULL content; may include <execute_python> tags)
full_content_round = generate_llm_code(
    prompt_round,
    inventory_tbl=inventory_tbl,
    transactions_tbl=transactions_tbl,
    model="o4-mini",
    temperature=1.0,
)

# Inspect the LLM’s plan + code (no execution here)
utils.print_html(full_content_round, title="Plan with Code (Full Response)")

### 2.4. Define the executor function (run a given plan)
The executor:
- accepts the full LLM response (with `<execute_python>` tags) or raw Python code,
- extracts the executable block,
- runs it in a controlled namespace (TinyDB tables + helpers only),
- captures stdout, errors and the answer variables (`answer_text`, `answer_rows`, `answer_json`).

> **One change from the lab:** the lab runs `exec(code, SAFE_GLOBALS, SAFE_LOCALS)`. With two separate dicts, any function or lambda the model defines can't see the code's own top-level imports and variables (→ `NameError`), which LLM-written code relies on a lot. Here the code runs in **one** namespace. The executor also returns the `STATUS` the plan sets.

> ⚠️ It's a *controlled* namespace, not a real security sandbox: the code can still import modules. Fine for this demo store; production systems run generated code in an isolated container.

In [ ]:
def _extract_execute_block(text: str) -> str:
    """
    Returns the Python code inside <execute_python>...</execute_python>.
    If no tags are found, assumes 'text' is already raw Python code.
    """
    if not text:
        raise RuntimeError("Empty content passed to code executor.")
    m = re.search(r"<execute_python>(.*?)</execute_python>", text, re.DOTALL | re.IGNORECASE)
    return m.group(1).strip() if m else text.strip()


def execute_generated_code(
    code_or_content: str,
    *,
    db,
    inventory_tbl,
    transactions_tbl,
    user_request: Optional[str] = None,
) -> Dict[str, Any]:
    """
    Execute code in a controlled namespace.
    Accepts either raw Python code OR full content with <execute_python> tags.
    Returns minimal artifacts: stdout, error, and extracted answer.
    """
    # Extract code here (now centralized)
    code = _extract_execute_block(code_or_content)

    SAFE_GLOBALS = {
        "Query": Query,
        "get_current_balance": inv_utils.get_current_balance,
        "next_transaction_id": inv_utils.next_transaction_id,
        "user_request": user_request or "",
    }
    SAFE_LOCALS = {
        "db": db,
        "inventory_tbl": inventory_tbl,
        "transactions_tbl": transactions_tbl,
    }
    # Change from the lab, which ran exec(code, SAFE_GLOBALS, SAFE_LOCALS): with two separate
    # dicts, functions and lambdas defined in the generated code cannot see its top-level
    # imports/variables (NameError). One shared namespace avoids that.
    namespace = {**SAFE_GLOBALS, **SAFE_LOCALS}

    # Capture stdout from the executed code
    _stdout_buf, _old_stdout = io.StringIO(), sys.stdout
    sys.stdout = _stdout_buf
    err_text = None
    try:
        exec(code, namespace)
    except Exception:
        err_text = traceback.format_exc()
    finally:
        sys.stdout = _old_stdout
    printed = _stdout_buf.getvalue().strip()

    # Extract possible answers set by the generated code
    answer = (
        namespace.get("answer_text")
        or namespace.get("answer_rows")
        or namespace.get("answer_json")
    )


    return {
        "code": code,            # code without the <execute_python> tags
        "stdout": printed,
        "error": err_text,
        "answer": answer,
        "status": namespace.get("STATUS"),  # addition: the STATUS the plan reported
        "transactions_tbl": transactions_tbl.all(),  # For inspection
        "inventory_tbl": inventory_tbl.all(),  # For inspection
    }

Hand the model's plan to the executor:

In [ ]:
# Execute the generated plan for the round-sunglasses question
result = execute_generated_code(
    full_content_round,          # the full LLM response you generated earlier
    db=db,
    inventory_tbl=inventory_tbl,
    transactions_tbl=transactions_tbl,
    user_request=prompt_round, # e.g., "Do you have any round sunglasses in stock that are under $100?"
)

# Peek at the answer the plan produced
utils.print_html(result["answer"], title="Plan Execution · Extracted Answer")

*(Extra)* The plan's `STATUS`, its stdout logs and any error:

In [ ]:
utils.print_html(f"STATUS: {result['status']}\n\n{result['stdout']}\n\n{result['error'] or ''}", title="Status, Logs & Errors")

### 2.4 Return Two Aviator Sunglasses
So far we only queried data. Now a **return**: *"Return 2 Aviator sunglasses I bought last week."*

Inventory before the return:

In [ ]:
Item = Query()                    # Create a Query object to reference fields (e.g., Item.name, Item.description)

# Query: fetch all inventory rows whose 'name' is exactly "Aviator".
# Notes:
# - This is a case-sensitive equality check. "aviator" won't match.
# - If you need case-insensitive matching, consider a .test(...) or .matches(...) with re.I.
aviators = inventory_tbl.search(
    (Item.name == "Aviator")
)

# Display the matched documents in a readable JSON panel
utils.print_html(json.dumps(aviators, indent=2), title="Inventory status: Aviator sunglasses before return")

One Aviator SKU — SG001: 23 units at $80 each. Generate the plan:

In [ ]:
prompt_aviator = "Return 2 Aviator sunglasses I bought last week."

# Generate the plan-as-code (FULL content; may include <execute_python> tags)
full_content_aviator = generate_llm_code(
    prompt_aviator,
    inventory_tbl=inventory_tbl,
    transactions_tbl=transactions_tbl,
    model="o4-mini",
    temperature=1,
)

# Inspect the LLM’s plan + code (no execution here)
utils.print_html(full_content_aviator, title="Plan with Code (Full Response)")

Transactions before executing the plan (just the $500 opening balance):

In [ ]:
utils.print_html(json.dumps(transactions_tbl.all(), indent=2), title="Transactions Table Before Return")

In [ ]:
# Execute the generated plan for the Aviator return
result = execute_generated_code(
    full_content_aviator,          # the full LLM response you generated earlier
    db=db,
    inventory_tbl=inventory_tbl,
    transactions_tbl=transactions_tbl,
    user_request=prompt_aviator, # e.g., "Return 2 aviator sunglasses I bought last week."
)

# Peek at the answer the plan produced
utils.print_html(result["answer"], title="Plan Execution · Extracted Answer")

A new transaction has been inserted for the return:

In [ ]:
utils.print_html(json.dumps(transactions_tbl.all(), indent=2), title="Transactions Table After Return")

And the Aviator stock is now 25:

In [ ]:
Item = Query()

aviators = inventory_tbl.search(
    (Item.name == "Aviator")
)

utils.print_html(json.dumps(aviators, indent=2), title="Inventory status: Aviator sunglasses after return")

## 3. Putting It All Together: Customer Service Agent
One helper that takes a natural-language request, generates a plan-as-code, executes it, and shows the answer plus before/after snapshots.

- Optionally reseeds the demo data for a clean run.
- Generates the plan (Python inside `<execute_python>…</execute_python>`).
- Executes the plan in a controlled namespace (TinyDB + helpers).
- Shows the concise `answer_text` and before/after tables.

In [ ]:
def customer_service_agent(
    question: str,
    *,
    db,
    inventory_tbl,
    transactions_tbl,
    model: str = "o4-mini",
    temperature: float = 1.0,
    reseed: bool = False,
) -> dict:
    """
    End-to-end helper:
      1) (Optional) reseed inventory & transactions
      2) Generate plan-as-code from `question`
      3) Execute in a controlled namespace
      4) Render before/after snapshots and return artifacts

    Returns:
      {
        "full_content": <raw LLM response (may include <execute_python> tags)>,
        "exec": {
            "code": <extracted python>,
            "stdout": <plan logs>,
            "error": <traceback or None>,
            "answer": <answer_text/rows/json>,
            "inventory_after": [...],
            "transactions_after": [...]
        }
      }
    """
    # 0) Optional reseed
    if reseed:
        inv_utils.create_inventory()
        inv_utils.create_transactions()

    # 1) Show the question
    utils.print_html(question, title="User Question")

    # 2) Generate plan-as-code (FULL content)
    full_content = generate_llm_code(
        question,
        inventory_tbl=inventory_tbl,
        transactions_tbl=transactions_tbl,
        model=model,
        temperature=temperature,
    )
    utils.print_html(full_content, title="Plan with Code (Full Response)")

    # 3) Before snapshots
    utils.print_html(json.dumps(inventory_tbl.all(), indent=2), title="Inventory Table · Before")
    utils.print_html(json.dumps(transactions_tbl.all(), indent=2), title="Transactions Table · Before")

    # 4) Execute
    exec_res = execute_generated_code(
        full_content,
        db=db,
        inventory_tbl=inventory_tbl,
        transactions_tbl=transactions_tbl,
        user_request=question,
    )

    # 5) After snapshots + final answer
    utils.print_html(exec_res["answer"], title="Plan Execution · Extracted Answer")
    utils.print_html(json.dumps(inventory_tbl.all(), indent=2), title="Inventory Table · After")
    utils.print_html(json.dumps(transactions_tbl.all(), indent=2), title="Transactions Table · After")

    # 6) Return artifacts
    return {
        "full_content": full_content,
        "exec": {
            "code": exec_res["code"],
            "stdout": exec_res["stdout"],
            "error": exec_res["error"],
            "answer": exec_res["answer"],
            "status": exec_res["status"],
            "inventory_after": inventory_tbl.all(),
            "transactions_after": transactions_tbl.all(),
        },
    }

## 4. Try It Out (with the Customer Service Agent)
Try these prompts:
1. Read-only (Andrew's example): *"Do you have any round sunglasses in stock that are under $100?"*
2. Mutation — return: *"Return 2 Aviator sunglasses."*
3. Mutation — purchase: *"Purchase 3 Wayfarer sunglasses for customer Alice."*
4. Mutation — multiple items: *"I want to buy 3 pairs of classic sunglasses and 1 pair of aviator."*

More to try: *"I want 3 Clubmaster sunglasses"* (only 2 in stock → `insufficient_stock`), *"Can you refurbish my frames?"* (`unsupported_intent`).

🔎 **`reseed=True`** resets the inventory to the default products and the transactions to the opening balance, for a clean, reproducible run. Use `reseed=False` to continue from the current state.

In [ ]:
prompt = "I want to buy 3 pairs of classic sunglasses and 1 pair of aviator sunglasses."

out = customer_service_agent(
    prompt,
    db=db,
    inventory_tbl=inventory_tbl,
    transactions_tbl=transactions_tbl,
    model="o4-mini",
    temperature=1.0,
    reseed=True,   # set False to keep current state of the inventory and the transactions
)

*(Extra)* Status, logs and errors of that run:

In [ ]:
utils.print_html(f"STATUS: {out['exec']['status']}\n\n{out['exec']['stdout']}\n\n{out['exec']['error'] or ''}", title="Status, Logs & Errors")

## 5. Takeaways
- **You let code be the plan.** Following the "code-as-action" idea, the model wrote Python that chains the steps (filter → compute → update), and you just ran it.
- **You skipped the brittle tool soup.** Instead of many tiny tools or JSON plans, Python/TinyDB gives the model a big, familiar toolbox that handles many query shapes with one prompt.
- **You kept runs controlled and visible.** Code ran in a restricted namespace, logs/errors were captured, and before/after tables show exactly what changed and why.